In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ============================================================
# Cell A — Setup (run first in any new session)
# ============================================================
from google.colab import drive
drive.mount('/content/drive')
!pip install -q timm

import os, json, random, copy, time, shutil
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as T
import torchvision.models as models
from torch.utils.data import DataLoader, Dataset
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
import timm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# ---- paths ----
TEA_LEAF_ROOT = "/content/drive/MyDrive/Tea Leaf Dataset"
OUT           = "/content/drive/MyDrive/TeaLeaf_DINOv2_Study"
RESULTS_DIR   = f"{OUT}/results"
FIGURES_DIR   = f"{OUT}/figures"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

# ---- dataset -> local disk ----
LOCAL_DATA = "/tmp/tea_leaf_local"
if not os.path.isdir(LOCAL_DATA):
    print("Copying dataset to local disk (~2 min)...")
    shutil.copytree(TEA_LEAF_ROOT, LOCAL_DATA)
    print(" Copied")
else:
    print(" Local cache exists")

# ---- classes ----
classes = sorted([d for d in os.listdir(LOCAL_DATA)
                  if os.path.isdir(os.path.join(LOCAL_DATA, d))])
NUM_CLASSES     = len(classes)
CLASS_TO_IDX    = {c: i for i, c in enumerate(classes)}
healthy_classes = [c for c in classes if 'healthy' in c.lower()]
print(f"Classes ({NUM_CLASSES}): {classes}")

# ---- DINOv1 ViT-S/16 ----
dv1_path = os.path.join(OUT, 'checkpoints', 'dino_vits16.pth')
if not os.path.exists(dv1_path):
    os.makedirs(os.path.dirname(dv1_path), exist_ok=True)
    !wget -q -O {dv1_path} https://dl.fbaipublicfiles.com/dino/dino_deitsmall16_pretrain/dino_deitsmall16_pretrain.pth

dino_v1 = timm.create_model('vit_small_patch16_224', pretrained=False, num_classes=0)
ckpt = torch.load(dv1_path, map_location='cpu')
sd = {k.replace('module.','').replace('backbone.','').replace('encoder.',''): v
      for k, v in ckpt.items() if 'head' not in k}
dino_v1.load_state_dict(sd, strict=False)
dino_v1.eval().to(device)
for p in dino_v1.parameters():
    p.requires_grad = False

# ---- attention hook ----
_dv1_store = {}
dv1_last_attn = dino_v1.blocks[-1].attn

def _dv1_patched_forward(x, **kwargs):
    B, N, C = x.shape
    qkv = dv1_last_attn.qkv(x).reshape(B, N, 3, dv1_last_attn.num_heads,
                                       C // dv1_last_attn.num_heads)
    qkv = qkv.permute(2, 0, 3, 1, 4)
    q, k, v = qkv.unbind(0)
    scale = (C // dv1_last_attn.num_heads) ** -0.5
    cls_q = q[:, :, 0:1, :]
    cls_attn = ((cls_q @ k.transpose(-2, -1)) * scale).softmax(dim=-1)
    _dv1_store['cls_attn'] = cls_attn[:, :, 0, 1:].detach().cpu()
    attn_full = ((q @ k.transpose(-2, -1)) * scale).softmax(dim=-1)
    x = (attn_full @ v).transpose(1, 2).reshape(B, N, C)
    x = dv1_last_attn.proj(x)
    x = dv1_last_attn.proj_drop(x)
    return x

dv1_last_attn.forward = _dv1_patched_forward

# ---- transforms ----
tf_dv1 = T.Compose([
    T.Resize((224, 224)), T.ToTensor(),
    T.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225]),
])
tf_train = T.Compose([
    T.Resize((224, 224)), T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
    T.RandomRotation(20), T.ColorJitter(0.2, 0.2, 0.2),
    T.ToTensor(), T.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225]),
])
tf_test = T.Compose([
    T.Resize((224, 224)), T.ToTensor(),
    T.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225]),
])

DV1_GRID = 14
N_HEADS  = 6
topk     = max(1, int(0.1 * DV1_GRID * DV1_GRID))

with torch.no_grad():
    _ = dino_v1(torch.randn(1, 3, 224, 224).to(device))
print(f"Attention hook OK: {_dv1_store['cls_attn'].shape}")

# ---- image list ----
all_images = []
for cls_name in classes:
    cls_dir = os.path.join(LOCAL_DATA, cls_name)
    for fname in sorted(os.listdir(cls_dir)):
        if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            all_images.append({
                'path': os.path.join(cls_dir, fname),
                'class': cls_name,
                'class_idx': CLASS_TO_IDX[cls_name],
                'is_diseased': 0 if cls_name in healthy_classes else 1,
            })

# ---- datasets ----
class SimpleDS(Dataset):
    def __init__(self, imgs, labels, transform):
        self.imgs = imgs; self.labels = labels; self.transform = transform
    def __len__(self): return len(self.imgs)
    def __getitem__(self, idx):
        img = Image.open(self.imgs[idx]['path']).convert('RGB')
        return self.transform(img), self.labels[idx]

class TestDS(Dataset):
    def __init__(self, imgs, transform):
        self.imgs = imgs; self.transform = transform
    def __len__(self): return len(self.imgs)
    def __getitem__(self, idx):
        img = Image.open(self.imgs[idx]['path']).convert('RGB')
        return self.transform(img), self.imgs[idx]['is_diseased']

n_dis = sum(x['is_diseased'] for x in all_images)
print(f"\n Ready: {len(all_images)} images | Diseased: {n_dis}, Healthy: {len(all_images)-n_dis}")

In [ ]:
# Upload annotate_tool.py and localization_eval.py
import os, shutil, sys
from google.colab import files

need = [f for f in ["annotate_tool.py", "localization_eval.py"]
        if not os.path.exists(f"/content/{f}")]

if need:
    print("Select BOTH files (ctrl/cmd-click to multi-select):", need)
    up = files.upload()
    for name in up:
        dest = f"/content/{os.path.basename(name)}"
        if os.path.abspath(name) != os.path.abspath(dest):
            shutil.move(name, dest)

for m in ["annotate_tool", "localization_eval"]:
    sys.modules.pop(m, None)          # clear any stale cache

import annotate_tool as at
import localization_eval as le
print("OK:", at.__file__, "|", le.__file__)

In [ ]:
for name in ['classes', 'all_images', 'dino_v1', 'tf_dv1', '_dv1_store',
             'DV1_GRID', 'RESULTS_DIR', 'device']:
    print(f"{name:<14s} {'OK' if name in dir() else 'MISSING — run Cell A'}")

In [ ]:
import random, json, os
import annotate_tool as at

random.seed(123)
DISEASE_CLASSES = [c for c in classes if 'healthy' not in c.lower()]   # 6 classes
PER_CLASS = 9                                                          # 6 x 9 = 54

sample = []
for c in DISEASE_CLASSES:
    pool = [x for x in all_images if x['class'] == c]
    for s in random.sample(pool, min(PER_CLASS, len(pool))):
        sample.append({'path': s['path'], 'class': c})
random.shuffle(sample)   # avoid annotating class-by-class (reduces bias)

ANN_PATH = os.path.join(RESULTS_DIR, 'lesion_annotations.json')
print(f"{len(sample)} images to annotate across {len(DISEASE_CLASSES)} disease classes")

In [ ]:
at.annotate(sample, ANN_PATH)

In [ ]:
import importlib, localization_eval as le
importlib.reload(le)
import numpy as np, torch
from PIL import Image

with open(ANN_PATH) as f:
    ann = json.load(f)
ann = {p: v for p, v in ann.items() if v['boxes']}     # drop skipped
print(f"Evaluating on {len(ann)} annotated images (skipped ones excluded)")

# use the SAME UAHS heads your pipeline selected
SEL_HEADS = [0, 1]

records = []
with torch.no_grad():
    for path, v in ann.items():
        im = Image.open(path).convert('RGB')
        _ = dino_v1(tf_dv1(im).unsqueeze(0).to(device))
        attn = _dv1_store['cls_attn'][0].numpy()               # [heads, 196]
        a = attn[SEL_HEADS].mean(axis=0).reshape(DV1_GRID, DV1_GRID)
        records.append({'attn': a, 'boxes': v['boxes'], 'class': v['class']})

res = le.evaluate(records, grid=DV1_GRID)
le.print_report(res)

m, lo, hi = le.bootstrap_ci(records, DV1_GRID, metric='pointing_game')
print(f"\nAttention − center (pointing game): {m:+.3f}  95% CI [{lo:+.3f}, {hi:+.3f}]")
print("CI excludes 0 -> attention localizes beyond center bias." if lo > 0
      else "CI includes 0 -> NOT distinguishable from center bias.")

with open(os.path.join(RESULTS_DIR, 'localization_validation.json'), 'w') as f:
    json.dump(res, f, indent=2)

In [ ]:
import numpy as np
from scipy.stats import wilcoxon
import localization_eval as le

gts = [le.boxes_to_mask(r['boxes'], DV1_GRID) for r in records]
cp  = le.center_prior(DV1_GRID)
prior = np.mean([g.astype(float) for g in gts], axis=0); prior /= prior.sum()

def per_image(fn, maps):
    return np.array([fn(a, g) for a, g in zip(maps, gts)])

A = [r['attn'] for r in records]
C = [cp] * len(records)
P = [prior] * len(records)

def compare(name, fn):
    a, c, p = per_image(fn, A), per_image(fn, C), per_image(fn, P)
    rng = np.random.default_rng(0); n = len(a)
    out = []
    for label, b in [("center", c), ("gt_prior", p)]:
        d = a - b
        boots = [d[rng.integers(0, n, n)].mean() for _ in range(5000)]
        lo, hi = np.percentile(boots, [2.5, 97.5])
        try:   _, pv = wilcoxon(a, b)
        except ValueError: pv = 1.0
        sig = "SIG" if lo > 0 else "n.s."
        out.append(f"    vs {label:<9s} Δ={d.mean():+.3f} CI[{lo:+.3f},{hi:+.3f}] "
                   f"p={pv:.2e}  {sig}")
    print(f"  {name}:  attention={a.mean():.3f}  center={c.mean():.3f}  prior={p.mean():.3f}")
    print("\n".join(out))

print("="*66); print("PAIRED TESTS (n=%d)" % len(records)); print("="*66)
compare("attn_mass_in", le.attn_mass_inside)
compare("iou@top10",   lambda a,g: le.iou_at_topk(a,g,0.10))
compare("iou@top20",   lambda a,g: le.iou_at_topk(a,g,0.20))

In [ ]:
print("records:", "OK" if 'records' in dir() else "MISSING — run V3 below")
print("dino_v1:", "OK" if 'dino_v1' in dir() else "MISSING — run Cell A first")

In [ ]:
import random, json, os
import annotate_tool as at

random.seed(123)
DISEASE_CLASSES = [c for c in classes if 'healthy' not in c.lower()]
PER_CLASS = 9

sample = []
for c in DISEASE_CLASSES:
    pool = [x for x in all_images if x['class'] == c]
    for s in random.sample(pool, min(PER_CLASS, len(pool))):
        sample.append({'path': s['path'], 'class': c})
random.shuffle(sample)

ANN_PATH = os.path.join(RESULTS_DIR, 'lesion_annotations.json')
print(f"{len(sample)} images | annotations at: {ANN_PATH}")

In [ ]:
import importlib, localization_eval as le
importlib.reload(le)
import numpy as np, torch, json
from PIL import Image

with open(ANN_PATH) as f:
    ann = json.load(f)
ann = {p: v for p, v in ann.items() if v['boxes']}
print(f"Evaluating on {len(ann)} annotated images")

SEL_HEADS = [0, 1]          # same heads UAHS selected

records = []
with torch.no_grad():
    for path, v in ann.items():
        im = Image.open(path).convert('RGB')
        _ = dino_v1(tf_dv1(im).unsqueeze(0).to(device))
        attn = _dv1_store['cls_attn'][0].numpy()
        a = attn[SEL_HEADS].mean(axis=0).reshape(DV1_GRID, DV1_GRID)
        records.append({'attn': a, 'boxes': v['boxes'], 'class': v['class']})

res = le.evaluate(records, grid=DV1_GRID)
le.print_report(res)

with open(os.path.join(RESULTS_DIR, 'localization_validation.json'), 'w') as f:
    json.dump(res, f, indent=2)
print("\nrecords ready — run V4")

In [ ]:
import numpy as np
from scipy.stats import wilcoxon
import localization_eval as le

gts   = [le.boxes_to_mask(r['boxes'], DV1_GRID) for r in records]
cp    = le.center_prior(DV1_GRID)
prior = np.mean([g.astype(float) for g in gts], axis=0); prior /= prior.sum()

A = [r['attn'] for r in records]
C = [cp] * len(records)
P = [prior] * len(records)

def per_image(fn, maps):
    return np.array([fn(a, g) for a, g in zip(maps, gts)])

def compare(name, fn):
    a, c, p = per_image(fn, A), per_image(fn, C), per_image(fn, P)
    rng = np.random.default_rng(0); n = len(a)
    print(f"\n  {name}:  attention={a.mean():.3f}  center={c.mean():.3f}  prior={p.mean():.3f}")
    for label, b in [("center", c), ("gt_prior", p)]:
        d = a - b
        boots = [d[rng.integers(0, n, n)].mean() for _ in range(5000)]
        lo, hi = np.percentile(boots, [2.5, 97.5])
        try:
            _, pv = wilcoxon(a, b)
        except ValueError:
            pv = 1.0
        sig = "SIG" if lo > 0 else "n.s."
        print(f"    vs {label:<9s} d={d.mean():+.3f}  CI[{lo:+.3f},{hi:+.3f}]  p={pv:.2e}  {sig}")

print("="*66)
print(f"PAIRED TESTS (n={len(records)})")
print("="*66)
compare("attn_mass_in", le.attn_mass_inside)
compare("iou@top10",   lambda a, g: le.iou_at_topk(a, g, 0.10))
compare("iou@top20",   lambda a, g: le.iou_at_topk(a, g, 0.20))

import os, json
np_out = {}
for nm, fn in [("attn_mass_in", le.attn_mass_inside),
               ("iou@top10", lambda a,g: le.iou_at_topk(a,g,0.10)),
               ("iou@top20", lambda a,g: le.iou_at_topk(a,g,0.20))]:
    a, c, p = per_image(fn, A), per_image(fn, C), per_image(fn, P)
    np_out[nm] = {"attention": float(a.mean()), "center": float(c.mean()),
                  "gt_prior": float(p.mean())}
with open(os.path.join(RESULTS_DIR, 'localization_paired_tests.json'), 'w') as f:
    json.dump(np_out, f, indent=2)
print("\nSaved -> localization_paired_tests.json")

In [ ]:
"""
fig1_pipeline.py -- Figure 1: zero-annotation pipeline workflow.

Vector (PDF) + raster (PNG) output, Elsevier-friendly:
  - single-column-safe line weights, 7-9pt type
  - TrueType fonts embedded (Type 42), not Type 3
  - greyscale-safe: stages differ by position/label, colour only reinforces
  - all reported numbers are parameters -> regenerate when results change

Usage:
    python fig1_pipeline.py                       # writes to ./
    build_figure(outdir="figures", **RESULTS)     # from a notebook
"""

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Rectangle
import os

# embed real fonts (Elsevier rejects Type 3)
matplotlib.rcParams["pdf.fonttype"] = 42
matplotlib.rcParams["ps.fonttype"] = 42
matplotlib.rcParams["font.family"] = "DejaVu Sans"

# muted, print-safe palette; distinguishable in greyscale by value
C_INPUT = "#E8EEF4"
C_FROZEN = "#D6E4F0"
C_SELECT = "#FBE7D0"
C_LABEL = "#E3EFE0"
C_TRAIN = "#F3DEE4"
C_OUT = "#DDD9EC"
EDGE = "#2F3B44"
ARROW = "#54636E"


def _box(ax, x, y, w, h, face, title, lines, title_size=8.4, body_size=7.0):
    ax.add_patch(FancyBboxPatch(
        (x, y), w, h, boxstyle="round,pad=0.012,rounding_size=0.02",
        linewidth=1.0, edgecolor=EDGE, facecolor=face, zorder=2))
    ax.text(x + w / 2, y + h - 0.052, title, ha="center", va="top",
            fontsize=title_size, fontweight="bold", color="#12181D", zorder=3)
    for i, ln in enumerate(lines):
        ax.text(x + w / 2, y + h - 0.115 - i * 0.052, ln, ha="center", va="top",
                fontsize=body_size, color="#2A343C", zorder=3)


def _arrow(ax, p0, p1, label=None, rad=0.0, ls="-"):
    ax.add_patch(FancyArrowPatch(
        p0, p1, arrowstyle="-|>", mutation_scale=11, linewidth=1.1,
        color=ARROW, linestyle=ls,
        connectionstyle=f"arc3,rad={rad}", zorder=1,
        shrinkA=1, shrinkB=1))
    if label:
        mx, my = (p0[0] + p1[0]) / 2, (p0[1] + p1[1]) / 2
        ax.text(mx, my + 0.028, label, ha="center", va="bottom",
                fontsize=6.6, style="italic", color="#4A565F", zorder=3)


def build_figure(outdir=".",
                 n_images=13084, n_classes=7,
                 backbone="DINOv1 ViT-S/16",
                 n_heads=6, sel_heads="2 of 6", grid="14x14",
                 pseudo_acc="89.6%",
                 auc="0.905 +/- 0.011", acc="0.907", f1="0.946",
                 student="MobileNetV3-Small",
                 params="1.52M", size_mb="6.1 MB", fps="100 FPS (CPU)",
                 compression="14.3x", speedup="3.1x",
                 fname="fig1_pipeline"):

    fig, ax = plt.subplots(figsize=(7.4, 4.15))
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")

    # ---- zero-annotation band (stages 1-4) -------------------------------
    ax.add_patch(Rectangle((0.012, 0.335), 0.700, 0.560, transform=ax.transData,
                           facecolor="#F7F9FB", edgecolor="#B9C6D0",
                           linewidth=0.9, linestyle=(0, (4, 2)), zorder=0))
    ax.text(0.362, 0.872, "NO HUMAN LABELS USED", ha="center", va="center",
            fontsize=7.4, fontweight="bold", color="#5A6A76", zorder=1,
            bbox=dict(boxstyle="round,pad=0.25", fc="#F7F9FB", ec="none"))

    y, h = 0.470, 0.330
    w = 0.152
    xs = [0.030, 0.207, 0.384, 0.545]

    _box(ax, xs[0], y, w, h, C_INPUT, "1. Unlabelled images",
         [f"{n_images:,} field photos", f"{n_classes} classes", "no annotation"])

    _box(ax, xs[1], y, w, h, C_FROZEN, "2. Frozen backbone",
         [backbone, "self-supervised", f"CLS attention {grid}"])

    _box(ax, xs[2], y, w, h, C_SELECT, "3. UAHS",
         ["rank heads by", "entropy variance", f"select {sel_heads}"])

    _box(ax, xs[3], y, 0.140, h, C_LABEL, "4. Pseudo-labels",
         ["top-10% attention", "mass -> threshold",
          f"acc {pseudo_acc}"])

    for i in range(3):
        _arrow(ax, (xs[i] + w, y + h / 2), (xs[i + 1], y + h / 2))
    _arrow(ax, (xs[2] + w, y + h / 2), (xs[3], y + h / 2))

    # ---- stage 5: student training --------------------------------------
    bx, by, bw, bh = 0.742, 0.470, 0.235, 0.330
    _box(ax, bx, by, bw, bh, C_TRAIN, "5. Train student",
         [student, "supervised on", "pseudo-labels only"])
    _arrow(ax, (xs[3] + 0.140, y + h / 2), (bx, by + bh / 2))

    # ---- deployed model + results ---------------------------------------
    dx, dy, dw, dh = 0.742, 0.075, 0.235, 0.300
    _box(ax, dx, dy, dw, dh, C_OUT, "Deployed classifier",
         [f"{params} params, {size_mb}", fps,
          f"{compression} smaller, {speedup} faster"])
    _arrow(ax, (bx + bw / 2, by), (dx + dw / 2, dy + dh))

    # ---- evaluation (held-out, true labels used ONLY here) ---------------
    ex, ey, ew, eh = 0.030, 0.075, 0.560, 0.220
    ax.add_patch(FancyBboxPatch(
        (ex, ey), ew, eh, boxstyle="round,pad=0.012,rounding_size=0.02",
        linewidth=1.0, edgecolor=EDGE, facecolor="#FFFFFF", zorder=2))
    ax.text(ex + 0.014, ey + eh - 0.045, "Evaluation (held-out test set)",
            ha="left", va="top", fontsize=8.2, fontweight="bold", color="#12181D")
    ax.text(ex + 0.014, ey + eh - 0.108,
            f"AUC {auc}     Accuracy {acc}     F1 {f1}",
            ha="left", va="top", fontsize=7.6, color="#1E2A33")
    ax.text(ex + 0.014, ey + eh - 0.163,
            "3 seeds - group-aware split (deduplicated, no train/test leakage)",
            ha="left", va="top", fontsize=6.8, style="italic", color="#5A6A76")

    _arrow(ax, (dx, dy + dh / 2), (ex + ew, ey + eh / 2), rad=0.0,
           label="predictions")
    ax.text(ex + ew - 0.014, ey + 0.030,
            "ground-truth labels are used ONLY for this evaluation",
            ha="right", va="bottom", fontsize=6.4, style="italic", color="#8A5A2B")

    os.makedirs(outdir, exist_ok=True)
    pdf = os.path.join(outdir, fname + ".pdf")
    png = os.path.join(outdir, fname + ".png")
    fig.savefig(pdf, bbox_inches="tight")
    fig.savefig(png, dpi=400, bbox_inches="tight")
    plt.close(fig)
    print("saved:", pdf, "|", png)
    return pdf, png


if __name__ == "__main__":
    build_figure(outdir=".")

In [ ]:
# export REAL panels for Figure 1
import numpy as np, matplotlib.pyplot as plt, torch, os
from PIL import Image

os.makedirs(f"{FIGURES_DIR}/fig1_panels", exist_ok=True)
sample = [x for x in all_images if x['class'] == 'Helopelties'][10]   # a clear focal lesion
im = Image.open(sample['path']).convert('RGB')

with torch.no_grad():
    _ = dino_v1(tf_dv1(im).unsqueeze(0).to(device))
attn = _dv1_store['cls_attn'][0].numpy()          # [6, 196] -> your REAL heads

im.resize((300,300)).save(f"{FIGURES_DIR}/fig1_panels/leaf.png")

# real mean attention heatmap
mean_a = attn[[0,1]].mean(0).reshape(14,14)
fig, ax = plt.subplots(figsize=(2.4,2.4))
h = ax.imshow(mean_a, cmap='Blues'); ax.set_xticks([]); ax.set_yticks([])
fig.colorbar(h, fraction=0.046)
fig.savefig(f"{FIGURES_DIR}/fig1_panels/attn_mean.pdf", bbox_inches='tight'); plt.close(fig)

# real per-head maps, heads 0 & 1 (UAHS-selected) outlined
fig, axes = plt.subplots(2, 3, figsize=(5.0, 3.4))
for h_i, ax in enumerate(axes.ravel()):
    ax.imshow(attn[h_i].reshape(14,14), cmap='Blues')
    ax.set_title(f"Head {h_i}", fontsize=8); ax.set_xticks([]); ax.set_yticks([])
    if h_i in (0, 1):
        for s in ax.spines.values():
            s.set_edgecolor('#E8792B'); s.set_linewidth(2.5)
fig.tight_layout()
fig.savefig(f"{FIGURES_DIR}/fig1_panels/heads.pdf", bbox_inches='tight'); plt.close(fig)
print("real panels saved")